# Extracting Geospatial Embeddings with Prithvi-EO-2.0
### Roma Norte, Mexico City

**Research internship (estancia de investigación) — Master's in Data Science, ITAM**
Student: Manuel Alonso De la Tejera González
Supervisor: Carlos López de la Cerda — Washington University in St. Louis

---

## Objective

This notebook documents the extraction of a geospatial embedding with
**Prithvi-EO-2.0** (NASA / IBM) for a Sentinel-2 chip of Roma Norte, CDMX, as a
third comparison point alongside **AlphaEarth (GSED)** and **Clay v1.5**, within
the internship's main project, *Earth Embedding Benchmarks for Geospatial
Prediction*.

Prithvi-EO-2.0 differs from both models in a meaningful way: it is a
**multi-temporal** Vision Transformer, pretrained with 3D (time, height, width)
patch embeddings on NASA's Harmonized Landsat Sentinel-2 (HLS) archive at 30m
resolution. It can take a sequence of several dates as input, not just one — this
notebook uses a single date for parity with the AlphaEarth and Clay tutorials, but
the time dimension is part of the model from the ground up, not an add-on.

The official tooling for Prithvi is **TerraTorch** (IBM's fine-tuning framework),
but the underlying ViT encoder can be used directly for embedding extraction
without setting up a full fine-tuning task — this notebook does exactly that.

## References

- Szwarcman et al. (2025). *Prithvi-EO-2.0: A Versatile Multi-Temporal Foundation
  Model for Earth Observation Applications*. arXiv:2412.02732
- Jakubik et al. (2023). *Foundation Models for Generalist Geospatial Artificial
  Intelligence*. arXiv:2310.18660
- NASA-IMPACT/Prithvi-EO-2.0: https://github.com/NASA-IMPACT/Prithvi-EO-2.0
- TerraTorch: https://github.com/IBM/terratorch
- Brown et al. (2025). *AlphaEarth Foundations*. arXiv:2507.22291


## 1. Environment setup

Unlike Clay, Prithvi does not require cloning a repository or manually downloading
a checkpoint: the official path is the **TerraTorch** library, which wraps weight
download (from Hugging Face) and model construction in a single call.

In [1]:
# Install TerraTorch and the Earth Engine client
!pip install terratorch -q
!pip install earthengine-api -q

### Google Earth Engine authentication

In [2]:
import ee
from google.colab import auth

auth.authenticate_user()
ee.Authenticate()
ee.Initialize(project='earth-embeddings-project')

print("GEE authenticated successfully")

GEE authenticated successfully


### Hardware check

Prithvi-EO-2.0-300M (the variant used below) has 300M parameters — smaller than
Clay's 632M, but a GPU still makes inference noticeably faster.

In [3]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## 2. Loading the Prithvi-EO-2.0 backbone

TerraTorch exposes pretrained backbones through a single registry call. We use
`prithvi_eo_v2_300` — the base encoder-only variant (1024-dimensional output, 24
transformer blocks), without the optional time/location conditioning ("TL"
variants), to keep this comparable to how we used Clay's and AlphaEarth's base
encoders.

`bands` tells the model which physical band each input channel corresponds to;
the model was pretrained on six HLS bands in this exact order — Blue, Green, Red,
Narrow NIR, SWIR 1, SWIR 2 — which for Sentinel-2 map to bands B2, B3, B4, B8A,
B11, B12. We pass it explicitly to be explicit about the mapping rather than
relying on the default order silently.

In [4]:
from terratorch.registry import BACKBONE_REGISTRY
from terratorch.datasets import HLSBands

model = BACKBONE_REGISTRY.build(
    "prithvi_eo_v2_300",
    pretrained=True,
    bands=[HLSBands.BLUE, HLSBands.GREEN, HLSBands.RED,
           HLSBands.NIR_NARROW, HLSBands.SWIR_1, HLSBands.SWIR_2],
)
model = model.to(device)
model.eval()

print(f"Model loaded on: {device}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Embedding dimension: {model.embed_dim}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Prithvi_EO_V2_300M.pt:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Model loaded on: cuda
Total parameters: 303,886,336
Embedding dimension: 1024


## 3. Downloading the satellite image

Same area of interest as the Clay notebook (Roma Norte, CDMX), but with a
different set of bands: Prithvi expects exactly the six HLS bands above, in that
order — `B2, B3, B4, B8A, B11, B12`.

We reuse the `ZIPPED_GEO_TIFF_PER_BAND` export format from the Clay notebook:
`NPY` flattens the bands into a single 2D array and loses the channel dimension,
so it does not work here either.

In [5]:
import ee
import numpy as np
import requests, io, zipfile
import rasterio

# Area of interest — Roma Norte, CDMX (same chip as the Clay notebook)
aoi = ee.Geometry.Rectangle([-99.165, 19.410, -99.150, 19.425])

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(aoi) \
    .filterDate('2022-01-01', '2022-12-31') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)) \
    .sort('CLOUDY_PIXEL_PERCENTAGE') \
    .first()

# Blue, Green, Red, NIR Narrow, SWIR1, SWIR2 — the six bands Prithvi expects
bands = ['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']
s2_clipped = s2.select(bands).clip(aoi)

url = s2_clipped.getDownloadURL({
    'scale': 10,
    'region': aoi,
    'format': 'ZIPPED_GEO_TIFF_PER_BAND',
    'bands': bands
})

response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))

arrays = []
for band in bands:
    fname = [f for f in z.namelist() if band in f][0]
    with rasterio.open(io.BytesIO(z.read(fname))) as src:
        arrays.append(src.read(1))

chip = np.stack(arrays, axis=0).astype(np.float32)
print(f"Chip shape: {chip.shape}")  # expected: (6, H, W)
print(f"Min: {chip.min():.1f}, Max: {chip.max():.1f}")

Chip shape: (6, 167, 159)
Min: 74.0, Max: 5402.0


Prithvi's patch embedding uses a `16×16` spatial patch size for this
variant (`prithvi_eo_v2_300`), with no padding — so height and width each need to
be an exact multiple of 16. We crop the chip to the nearest valid square size,
the same way we did for Clay (which needed a multiple of 8).

In [6]:
patch_size = 16
size = (min(chip.shape[1], chip.shape[2]) // patch_size) * patch_size
top = (chip.shape[1] - size) // 2
left = (chip.shape[2] - size) // 2
chip = chip[:, top:top+size, left:left+size]

print(f"Cropped chip shape: {chip.shape}")

Cropped chip shape: (6, 144, 144)


## 4. Preprocessing: per-band normalization

TerraTorch hardcodes the per-band mean and standard deviation used to pretrain
each Prithvi variant directly in its source (`prithvi_vit.py`); they are not
applied automatically inside the model, so we standardize the chip ourselves
before passing it in — the same pattern as Clay's `metadata.yaml` values, just
sourced from a different place.

These are the values for Prithvi-EO-2.0, in the same band order as the chip
(Blue, Green, Red, NIR Narrow, SWIR 1, SWIR 2):

In [7]:
import torch

PRITHVI_V2_MEAN = [1087.0, 1342.0, 1433.0, 2734.0, 1958.0, 1363.0]
PRITHVI_V2_STD  = [2248.0, 2179.0, 2178.0, 1850.0, 1242.0, 1049.0]

means = torch.tensor(PRITHVI_V2_MEAN).view(-1, 1, 1)
stds  = torch.tensor(PRITHVI_V2_STD).view(-1, 1, 1)

chip_t = torch.from_numpy(chip).float()   # (6, H, W)
chip_norm = (chip_t - means) / stds       # per-band standardization

# Prithvi's encoder accepts a plain (B, C, H, W) tensor and adds the time axis
# internally when num_frames=1 — no manual unsqueeze for time needed here.
chip_tensor = chip_norm.unsqueeze(0)      # (1, 6, H, W)

print(f"Tensor shape: {chip_tensor.shape}")
print(f"Min: {chip_tensor.min():.3f}, Max: {chip_tensor.max():.3f}")

Tensor shape: torch.Size([1, 6, 144, 144])
Min: -1.438, Max: 2.758


## 5. Extracting the embedding

Because the backbone was built with `pretrained=True` and no fine-tuning head
(`encoder_only=True` is the default), calling the model directly returns the
hidden states from every transformer block as a list, without any random
masking — masking is only applied during the original MAE pretraining, not here.
We take the **last** block's output, which has already passed through the final
layer norm, and read off the CLS token (position 0) as the chip's embedding.

In [8]:
with torch.no_grad():
    features = model(chip_tensor.to(device))   # list of per-block hidden states

last_layer = features[-1]                       # (1, 1 + num_patches, embed_dim)
embedding = last_layer[:, 0, :].squeeze().cpu().numpy()  # CLS token

print(f"Embedding shape: {embedding.shape}")  # expected: (1024,)
print("First 5 dimensions:")
for i in range(5):
    print(f"  dim_{i:03d}: {embedding[i]:.6f}")

Embedding shape: (1024,)
First 5 dimensions:
  dim_000: -0.168140
  dim_001: -0.062317
  dim_002: -0.057786
  dim_003: 0.003314
  dim_004: -0.151565


## 6. Discussion and next steps

A few points worth carrying into the internship report:

- **Dimensionality.** `prithvi_eo_v2_300` also outputs 1024 dimensions — the same
  as Clay, and 16x AlphaEarth's 64. Dimensionality alone does not distinguish
  these two models; what differs is what each backbone was trained on (Sentinel-2
  directly for Clay vs. the harmonized Landsat/Sentinel-2 archive for Prithvi)
  and the temporal design (Prithvi treats time as a first-class input dimension).
- **Multi-temporal design, unused here.** This notebook passes a single date,
  but Prithvi's architecture is built around sequences of dates (`num_frames` can
  be set above 1). A natural extension for the internship is comparing a
  single-date embedding against a multi-date one for the same AGEB, to see
  whether seasonal variation adds predictive signal beyond a single snapshot —
  something AlphaEarth's annual composites and Clay's single-image design cannot
  directly test.
- **Implementation robustness.** Clay's positional encoding assumes a square
  patch grid internally, which broke when our chip wasn't square (Section 3 of
  the Clay notebook). Prithvi's position-encoding interpolation handles height
  and width independently, so a non-square chip is not an issue here beyond the
  multiple-of-`patch_size` requirement. Worth a line in the "panorama de modelos"
  section of the report — implementation maturity varies across these models,
  not just architecture.
- **Computational cost.** Like Clay, Prithvi requires downloading imagery and
  running a forward pass per location — there is no precomputed database like
  AlphaEarth's GSED. At 300M parameters it is lighter than Clay's 632M, but still
  considerably more expensive at city scale than querying a precomputed tile.
